In [ ]:
!pip install chembl_webresource_client

from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np

# 1. Target auswählen
target_id = "CHEMBL203"  # EGFR

activity = new_client.activity
res = activity.filter(target_chembl_id=target_id, standard_type="IC50")

df = pd.DataFrame(res)

# 2. Nur relevante Spalten behalten
df = df[["canonical_smiles", "standard_value"]].dropna()

# 3. standard_value in numerisch umwandeln
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

# 4. Ungültige Werte entfernen
df = df[df["standard_value"] > 0]

# 5. IC50 (nM) → pIC50
df["pIC50"] = -np.log10(df["standard_value"] * 1e-9)

# 6. Speichern
df.to_csv("egfr_raw.csv", index=False)

print("Saved egfr_raw.csv with shape:", df.shape)



Saved egfr_raw.csv with shape: (24344, 3)


In [ ]:
!pip install rdkit
from rdkit import Chem
from rdkit.Chem import MACCSkeys
import numpy as np
import pandas as pd

df = pd.read_csv("egfr_raw.csv")

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

df["fingerprint"] = df["canonical_smiles"].apply(smiles_to_fp)
df = df.dropna()

df.to_csv("egfr_processed.csv", index=False)
print("Saved egfr_processed.csv with shape:", df.shape)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 48.3 MB/s eta 0:00:00
Saved egfr_processed.csv with shape: (24344, 4)
